# Observed populations

This notebook reproduces the number of detected neutron stars in Equations 11 and 12 in the paper Pardo et al. (2025).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib import rc
import matplotlib as mpl
from scipy.integrate import quad
from scipy import stats

from mlpoppyns.simulator.config_simulator import cfg
import mlpoppyns.simulator.basics.constants as const
import utilities.plot_settings

from matplotlib import rc

rc("text", usetex=True)
rc("font", family="serif")
mpl.rcParams["text.latex.preamble"] = r"\usepackage{amsmath}"

In [ ]:
SMALL_SIZE = 30
MEDIUM_SIZE = 40
BIGGER_SIZE = 60

plt.rc("font", size=SMALL_SIZE)  # controls default text sizes
plt.rc("axes", titlesize=MEDIUM_SIZE)  # fontsize of the axes title
plt.rc("axes", labelsize=MEDIUM_SIZE)  # fontsize of the x and y labels
plt.rc("xtick", labelsize=MEDIUM_SIZE)  # fontsize of the tick labels
plt.rc("ytick", labelsize=MEDIUM_SIZE)  # fontsize of the tick labels
plt.rc("legend", fontsize=SMALL_SIZE)  # legend fontsize
plt.rc("figure", titlesize=BIGGER_SIZE)  # fontsize of the figure title

## Load observed data

Load the ATNF Pulsar Catalogue v2.4.0.

In [ ]:
# Read the full ATNF catalog.csv file. Binary pulsars are excluded.
df_atnf = pd.read_csv(
    "../../data/observations/atnf_full_nobinary_24-09-2024_with_errors.csv",
    delimiter=";",
    header=[0, 1],
)
df_atnf.head()

Load TPA program's MeerKAT data from Posselt et al. (2023).

In [ ]:
df_meerkat = df_meerkat_1 = pd.read_csv(
    "../../data/observations/meerkat_tpa_posselt_2023.csv",
    delimiter=",",
)

In [ ]:
# Select only those with measured period values.
df_atnf = df_atnf[~df_atnf["P0"]['(s)'].isin(["NAN"])]
#df_atnf['ASSOC'] = df_atnf['ASSOC'].fillna('').astype(str)

# Remove those objects that are in globular clusters or in the Magellanic Clouds.
discard = [
    "EXGAL:SMC",
    "EXGAL:LMC",
    "GC:47Tuc",
    "GC:M3",
    "GC:M5",
    "GC:M13",
    "GC:NGC6440",
    "GC:Ter5",
    "GC:NGC6441",
    "GC:NGC6517",
    "GC:NGC6522",
    "GC:NGC6624",
    "GC:M28(NGC6626)",
    "GC:NGC6652",
    "GC:M22(NGC6656)",
    "GC:NGC6752",
    "GC:NGC6760",
    "GC:M15",
    "GC:M30",
]
df_atnf = df_atnf[
    ~df_atnf[('ASSOC', 'Unnamed: 55_level_1')].str.match("|".join(discard))
]

len(df_atnf)

In [ ]:
df_atnf = df_atnf.drop(
    columns=[
        "#",
        "PMRA",
        "PMDEC",
        "PX",
        "POSEPOCH",
        "RAJD",
        "DECJD",
        "DM",
        "TAU_SC",
        "S400",
        "S2000",
        "DIST",
        "ZZ",
        "XX",
        "YY",
    ],
    level = 0
)

len(df_atnf)

In [ ]:
df_atnf.columns = df_atnf.columns.droplevel(1)

In [ ]:
# Select only isolated non-recycled neutron stars through filters with P > 0.01 and Pdot > 1e-19 (for those with measured values).
df_atnf = df_atnf[df_atnf["P0"].to_numpy().astype(np.float64) > 0.01]

len(df_atnf)

Note: For the purpose of modelling the observed isolated population of radio pulsars, in the following, we count those objects with period derivatives larger than $\dot{P} > 10^{-19} s/s$ or those with no measured period derivatives. The latter are likely isolated in nature due to the fact that only a small fraction of ATNF pulsars with known $\dot{P}$ actually have been recycled and attain $\dot{P} < 10^{-19} s/s$. As a result, the following number counts are slighlty larger than the populations used to produce our period period-derivative maps for the simulation-based inference approach.

In [ ]:
df_atnf = df_atnf[
    (df_atnf["P1"].to_numpy().astype(np.float64) > 1.0e-19)
    | (df_atnf["P1"].isin(["NAN"]))
]

In [ ]:
# Parkes Multibeam Pulsar Survey (PMPS) database.
df_atnf_pmps = df_atnf[
    df_atnf["SURVEY"].str.contains("pksmb")
]

l_pmps_obs = df_atnf_pmps["Gl"].to_numpy().astype(np.float64)
b_pmps_obs = df_atnf_pmps["Gb"].to_numpy().astype(np.float64)
P_pmps_obs = df_atnf_pmps["P0"].to_numpy().astype(np.float64)
Pdot_pmps_obs = df_atnf_pmps["P1"].to_numpy().astype(np.float64)
S1400_pmps_obs = df_atnf_pmps["S1400"].to_numpy().astype(np.float64)

# Convert galactic latitude in the range [-180., 180].
l_pmps_obs[(l_pmps_obs > 180.0) & (l_pmps_obs < 360.0)] = (
    l_pmps_obs[(l_pmps_obs > 180.0) & (l_pmps_obs < 360.0)] - 360.0
)

# Select only pulsars falling in the PMPS sky coverage where completness is above 90%.
cond = (l_pmps_obs > -100.0) & (l_pmps_obs < 50.0) & (np.abs(b_pmps_obs) < 5.0)

# Merge the MeerKAT TPA program data with the PMPS ATNF Pulsar Catalogue data to obtain MeerKAT flux measurements
# for the PMPS pulsars.
df_meerkat_pmps = pd.merge(
    df_meerkat, df_atnf_pmps[cond], left_on="PSRJ", right_on="PSRJ"
)
df_meerkat_pmps = df_meerkat_pmps.dropna(subset=["ch6flux"])

l_pmps_obs = l_pmps_obs[cond]
b_pmps_obs = b_pmps_obs[cond]
P_pmps_obs = P_pmps_obs[cond]
Pdot_pmps_obs = Pdot_pmps_obs[cond]
S1400_pmps_obs = S1400_pmps_obs[cond]

number_pmps = len(l_pmps_obs)

In [ ]:
# Swinburne Intermediate-latitude Pulsar Survey (SMPS) database.
df_atnf_smps = df_atnf[
    df_atnf["SURVEY"].str.contains("pkssw")
]

l_smps_obs = df_atnf_smps["Gl"].to_numpy().astype(np.float64)
b_smps_obs = df_atnf_smps["Gb"].to_numpy().astype(np.float64)
P_smps_obs = df_atnf_smps["P0"].to_numpy().astype(np.float64)
Pdot_smps_obs = df_atnf_smps["P1"].to_numpy().astype(np.float64)
S1400_smps_obs = df_atnf_smps["S1400"].to_numpy().astype(np.float64)

# Convert galactic latitude in the range [-180., 180].
l_smps_obs[(l_smps_obs > 180.0) & (l_smps_obs < 360.0)] = (
    l_smps_obs[(l_smps_obs > 180.0) & (l_smps_obs < 360.0)] - 360.0
)

# Selection only pulsars falling in the SMPS sky coverage where completness is above 90%.
cond = (l_smps_obs > -100.0) & (l_smps_obs < 50.0)

# Merge the MeerKAT TPA program data with the SMPS ATNF Pulsar Catalogue data to obtain MeerKAT flux measurements
# for the SMPS pulsars.
df_meerkat_smps = pd.merge(
    df_meerkat, df_atnf_smps[cond], left_on="PSRJ", right_on="PSRJ"
)
df_meerkat_smps = df_meerkat_smps.dropna(subset=["ch6flux"])

l_smps_obs = l_smps_obs[cond]
b_smps_obs = b_smps_obs[cond]
P_smps_obs = P_smps_obs[cond]
Pdot_smps_obs = Pdot_smps_obs[cond]
S1400_smps_obs = S1400_smps_obs[cond]

l_all_obs = np.concatenate((l_pmps_obs, l_smps_obs))
b_all_obs = np.concatenate((b_pmps_obs, b_smps_obs))
P_all_obs = np.concatenate((P_pmps_obs, P_smps_obs))
Pdot_all_obs = np.concatenate((Pdot_pmps_obs, Pdot_smps_obs))
S1400_all_obs = np.concatenate((S1400_pmps_obs, S1400_smps_obs))

number_smps = len(l_smps_obs)

In [ ]:
# Low- and mid-latitude High Time Resolution Universe (HTRU) database.
df_atnf_htru = df_atnf[
    df_atnf["SURVEY"].str.contains("htru_pks")
]

l_htru_obs = df_atnf_htru["Gl"].to_numpy().astype(np.float64)
b_htru_obs = df_atnf_htru["Gb"].to_numpy().astype(np.float64)
P_htru_obs = df_atnf_htru["P0"].to_numpy().astype(np.float64)
Pdot_htru_obs = df_atnf_htru["P1"].to_numpy().astype(np.float64)
S1400_htru_obs = df_atnf_htru["S1400"].to_numpy().astype(np.float64)

# Convert galactic latitude in the range [-180., 180].
l_htru_obs[(l_htru_obs > 180.0) & (l_htru_obs < 360.0)] = (
    l_htru_obs[(l_htru_obs > 180.0) & (l_htru_obs < 360.0)] - 360.0
)

# Selection only pulsars falling in the HTRU sky coverage where completness is above 90%.
cond = (
    (l_htru_obs > -120.0) & (l_htru_obs < 30.0) & (np.abs(b_htru_obs) < 15.0)
)

# Merge the MeerKAT TPA program data with the HTRU ATNF Pulsar Catalogue data to obtain MeerKAT flux measurements
# for the HTRU pulsars.
df_meerkat_htru = pd.merge(
    df_meerkat, df_atnf_htru, left_on="PSRJ", right_on="PSRJ"
)
df_meerkat_htru = df_meerkat_htru.dropna(subset=["ch6flux"])

l_htru_obs = l_htru_obs[cond]
b_htru_obs = b_htru_obs[cond]
P_htru_obs = P_htru_obs[cond]
Pdot_htru_obs = Pdot_htru_obs[cond]
S1400_htru_obs = S1400_htru_obs[cond]

l_all_obs = np.concatenate((l_pmps_obs, l_smps_obs, l_htru_obs))
b_all_obs = np.concatenate((b_pmps_obs, b_smps_obs, b_htru_obs))
P_all_obs = np.concatenate((P_pmps_obs, P_smps_obs, P_htru_obs))
Pdot_all_obs = np.concatenate((Pdot_pmps_obs, Pdot_smps_obs, Pdot_htru_obs))
S1400_all_obs = np.concatenate(
    (S1400_pmps_obs, S1400_smps_obs, S1400_htru_obs)
)

number_htru = len(l_htru_obs)

In [ ]:
print(f"Number of pulsars detected by PMPS: {number_pmps}")
print(f"Number of pulsars detected by SMPS: {number_smps}")
print(f"Number of pulsars detected by HTRU: {number_htru}")

In [ ]:
print(f"Number of pulsars overlapping between PMPS and MeerKAT: {len(df_meerkat_pmps)}")
print(f"Number of pulsars overlapping between SMPS and MeerKAT: {len(df_meerkat_smps)}")
print(f"Number of pulsars overlapping between HTRU and MeerKAT: {len(df_meerkat_htru)}")

In [ ]:
print(
    f"Number of pulsars detected by PMPS without Pdot measurement: {np.isnan(Pdot_pmps_obs).sum()}"
)
print(
    f"Number of pulsars detected by SMPS multibeam without Pdot measurement: {np.isnan(Pdot_smps_obs).sum()}"
)
print(
    f"Number of pulsars detected by HTRU without Pdot measurement: {np.isnan(Pdot_htru_obs).sum()}"
)

In [ ]:
print(
    f"Number of pulsars detected by PMPS without S1400 measurement: {np.isnan(S1400_pmps_obs).sum()}"
)
print(
    f"Number of pulsars detected by SMPS multibeam without S1400 measurement: {np.isnan(S1400_smps_obs).sum()}"
)
print(
    f"Number of pulsars detected by HTRU without S1400 measurement: {np.isnan(S1400_htru_obs).sum()}"
)

Comparison between the S1400 flux values from the ATNF catalog and the ch6flux column from MeerKAT.

In [ ]:
x = np.logspace(-2,5,1000)

fig, ax = plt.subplots(figsize=(15, 8))

ax.plot(
    df_meerkat_pmps['S1400'].to_numpy().astype(np.float64),
    df_meerkat_pmps['ch6flux'],
    linestyle="None",
    marker="o",
    color="tab:green",
    markersize=6,
    alpha=1.0,
    rasterized=True,
)
ax.plot(
    x,
    x,
    alpha=1.0,
    rasterized=True,
)
plt.xlabel(r"ATNF-PMPS flux")
plt.grid()
plt.ylabel(r"TPA-PMPS flux")
ax.set_xscale("log")
ax.set_yscale("log")


plt.show()

In [ ]:
x = np.logspace(-2,5,1000)

fig, ax = plt.subplots(figsize=(15, 8))

ax.plot(
    df_meerkat_smps['S1400'].to_numpy().astype(np.float64),
    df_meerkat_smps['ch6flux'],
    linestyle="None",
    marker="o",
    color="tab:green",
    markersize=6,
    alpha=1.0,
    rasterized=True,
)
ax.plot(
    x,
    x,
    alpha=1.0,
    rasterized=True,
)
plt.xlabel(r"ATNF-SMPS flux")
plt.grid()
plt.ylabel(r"TPA-SMPS flux")
ax.set_xscale("log")
ax.set_yscale("log")


plt.show()

In [ ]:
x = np.logspace(-2,5,1000)

fig, ax = plt.subplots(figsize=(15, 8))

ax.plot(
    df_meerkat_htru['S1400'].to_numpy().astype(np.float64),
    df_meerkat_htru['ch6flux'],
    linestyle="None",
    marker="o",
    color="tab:green",
    markersize=6,
    alpha=1.0,
    rasterized=True,
)
ax.plot(
    x,
    x,
    alpha=1.0,
    rasterized=True,
)
plt.xlabel(r"ATNF-HTRU flux")
plt.grid()
plt.ylabel(r"TPA-HTRU flux")
ax.set_xscale("log")
ax.set_yscale("log")


plt.show()

The generated plots compare the S1400 flux measurements from the ATNF catalog with the ch6 fluxes from MeerKAT, with the aim of determining whether the ch6 column in MeerKAT represents period-averaged fluxes and is therefore comparable to the S_obs_mean_S1400 values from the simulated population. Additionally, Posselt et al. (2023) use similar plots to demonstrate that while MeerKAT's flux measurements are consistent with those in the ATNF catalog, they offer greater accuracy due to MeerKAT's enhanced sensitivity.